In [1]:
%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

from IPython.display import clear_output

import json
import yaml

import os
import subprocess
import sys
import time

import tempfile
import h5py

import pandas as pd
import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import h5py

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['olivedrab', 'steelblue', 'firebrick', 'goldenrod'])
plt.rcParams['axes.formatter.use_mathtext'] = True
plt.rcParams['axes.formatter.useoffset'] = False
plt.rcParams['axes.formatter.limits'] = (0, 0)
plt.rcParams['figure.figsize'] = [6,4]
plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['legend.frameon'] = False
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['ytick.minor.visible'] = True

template_file = 'PSLS/examples/psls.yaml'

with open(template_file, 'r') as file:
    config_dict = yaml.safe_load(file)
    print(json.dumps(config_dict, indent=4, sort_keys=False))

{
    "Observation": {
        "QuarterDuration": [
            90.0,
            90.0,
            90.0
        ],
        "MasterSeed": 1704040900,
        "Gaps": {
            "Enable": 1,
            "Seed": -1,
            "InterQuarterGapDuration": 3.0,
            "RandomGapDuration": 0.0,
            "RandomGapTimeFraction": 0.5,
            "RandomGapStep": 0.0,
            "PeriodicGapCadence": 5.0,
            "PeriodicGapDuration": 20.0,
            "PeriodicGapJitter": 2.0,
            "PeriodicGapStep": 0.0
        }
    },
    "Instrument": {
        "Sampling": 25.0,
        "IntegrationTime": 21.0,
        "GroupID": [
            1,
            2,
            3,
            4
        ],
        "NCamera": 6,
        "TimeShift": 6.25,
        "RandomNoise": {
            "Enable": 1,
            "Type": "PLATO_SIMU",
            "NSR": 73.0
        },
        "Systematics": {
            "Enable": 1,
            "Table": "systematics/PLATO_systematics_BOL_V2.npy",
  

In [2]:
psls_template = {
    'Observation': {
        'QuarterDuration': [90.0, 90.0, 90.0],
        'MasterSeed': 1704040900,
        'Gaps': {
            'Enable': 1,
            'Seed': -1,
            'InterQuarterGapDuration': 3.0,
            'RandomGapDuration': 0.0,
            'RandomGapTimeFraction': 0.5,
            'RandomGapStep': 0.0,
            'PeriodicGapCadence': 5.0,
            'PeriodicGapDuration': 20.0,
            'PeriodicGapJitter': 2.0,
            'PeriodicGapStep': 0.0
        }
    },
    'Instrument': {
        'Sampling': 25.0,
        'IntegrationTime': 21.0,
        'GroupID': [
            1,
            2,
            3,
            4
        ],
        'NCamera': 6,
        'TimeShift': 6.25,
        'RandomNoise': {
            'Enable': 1,
            'Type': 'PLATO_SIMU',
            'NSR': 73.0
        },
        'Systematics': {
            'Enable': 1,
            'Table': 'systematics/PLATO_systematics_BOL_V2.npy',
            'Version': 2,
            'DriftLevel': 'any',
            'Seed': -1
        }
    },
    'Star': {
        'Mag': 10.0,
        'ID': 12069449,
        'ModelType': 'single',
        'ModelDir': 'models/',
        'ModelName': '0012069449',
        'ES': 'ms',
        'Teff': 5750.0,
        'Logg': 4.353,
        'SurfaceRotationPeriod': 0.0,
        'CoreRotationFreq': 0.0,
        'Inclination': 0.0
    },
    'Oscillations': {
        'Enable': 1,
        'numax': 179.3,
        'delta_nu': 13.68,
        'DPI': 80.58,
        'q': 0.15,
        'SurfaceEffects': 1,
        'Seed': -1
    },
    'Activity': {
        'Enable': 1,
        'Sigma': 40.0,
        'Tau': 0.2,
        'Seed': -1,
        'Spot': {
            'Enable': 0,
            'dOmega': 0.0,
            'MuStar': 0.59,
            'MuSpot': 0.78,
            'Radius': [2.5, 2.5, 2.5],
            'Latitude': [0.0, 20.0, 40.0],
            'Longitude': [0.0, 0.0, 0.0],
            'Lifetime': [10, 30, 50],
            'TimeMax': [-1, -1, -1],
            'Contrast': [0.7, 0.8, 0.6],
            'Modulation': 0.0,
            'Seed': -1
        },
        'Flare': {
            'Enable': 0,
            'MeanPeriod': 2,
            'Amplitude': 2500.0,
            'UpDown': 0.1,
            'MeanDuration': -1,
            'DurationDispersion': -1,
            'Seed': -1
        }
    },
    'Granulation': {
        'Enable': 1,
        'Type': 1,
        'Seed': -1
    },
    'Transit': {
        'Enable': 1,
        'PlanetRadius': 0.5,
        'OrbitalPeriod': 10.0,
        'PlanetSemiMajorAxis': 1.0,
        'OrbitalAngle': 0.0,
        'LimbDarkeningCoefficients': [0.25, 0.75]
    },
    'External': {
        'Enable': 0,
        'FilePath': 'examples/external_example.txt'
    }
}

In [ ]:
config_files = [psls_template, psls_template, psls_template, psls_template, psls_template]
start_batch = time.time()

total_sims = len(config_files)

initial_status = f'│ progress |{'░' * 50}| 0% [running 1/{total_sims}] (00:00:00/--:--:--)          '

print(f'{initial_status}')

for i, cfg in enumerate(config_files):

    if i != 0:
        perc = i / total_sims * 100
        bar = '█' * int(perc // 2) + '░' * (50 - int(perc // 2))
        status = f'│ progress |{bar}| {perc:.0f}% [running {i+1}/{total_sims}] ({elapsed_str}/{estimated_str})          '

        clear_output(wait=True)
        print(f'{status}')

    with tempfile.NamedTemporaryFile(suffix='.yaml', dir='PSLS', mode='w', delete=True) as tf:
        yaml.dump(cfg, tf, default_flow_style=False, sort_keys=False)
        tf.flush()

        sys.stdout.write(f'└── initializing simulation {i + 1}          ')
        sys.stdout.flush()
        time.sleep(1)
        sys.stdout.write(f'\r└─┬ initializing simulation {i + 1}          \n')
        sys.stdout.write(f'  └── sampling parameters [active]          ')
        sys.stdout.flush()
        time.sleep(1)
        sys.stdout.write(f'\r  ├── sampling parameters [done]          \n')
        sys.stdout.write(f'  └── generating lightcurve [active]          ')
        sys.stdout.flush()
        
        %cd -q PSLS
        !echo '' | ./psls.py -o data {tf.name}
        %cd -q ..

        sys.stdout.write(f'\r  ├── generating lightcurve [done]          \n')
        sys.stdout.write(f'  └── saving dataframe [active]          ')
        sys.stdout.flush()
        time.sleep(1)
        sys.stdout.write(f'\r  └── saving dataframe [done]          \n')
        sys.stdout.flush()
        time.sleep(1)

    current_run = time.time()
    
    elapsed_batch = current_run - start_batch
    
    average_time = elapsed_batch / (i + 1)
    
    estimated_batch = elapsed_batch + (total_sims - (i + 1)) * average_time

    elapsed_str = time.strftime('%H:%M:%S', time.gmtime(elapsed_batch))
    estimated_str = time.strftime('%H:%M:%S', time.gmtime(estimated_batch))

final_status = f'│ progress |{'█' * 50}| 100% [finished {i+1}/{total_sims}] ({elapsed_str}/{estimated_str})          '

clear_output(wait=True)
print(f'{final_status}')

│ progress |██████████████████████████████░░░░░░░░░░░░░░░░░░░░| 60% [running 4/5] (00:00:27/00:00:46)          
└─┬ initializing simulation 4          
  ├── sampling parameters [done]            
  ├── generating lightcurve [done]            
  └── saving dataframe [done]            
